### 적절한 하이퍼파라미터 값 찾기

- 하이퍼파라미터 예
- 각 층의 뉴런 수, 배치 크기, 매개변수 갱신 시의 학습률과 가중치 감소 등

- 하이퍼파라미터의 값을 적절히 설정하지 않으면 모델의 성능이 크게 떨어지기도 함


#### 검증 데이터

- 지금까지는 데이터셋을 훈련 데이터와 시험 데이터라는 두 가지로 분리해 이용

- 훈련 데이터로는 학습
- 시험 데이터로는 범용 성능 평가
- 훈련 데이터에만 지나치게 적응되어 있지 않은지(오버피팅한 건 아닌지)
- 범용 성능은 어느 정도인지 등을 평가 가능

- 하이퍼파라미터를 다양한 값으로 설정하고 검증할 때

- 하이퍼파라미터의 성능을 평가할 때는 시험 데이터를 사용해서 안됨

- 같은 성능 평가인데 왜 하이퍼파라미터가 대상일 때는 시험 데이터를 사용하면 안될까?

- 시험 데이터를 사용하여 하이퍼파라미터를 조정하면 하이퍼파라미터 값이 시험 데이터에 오버피팅되기 때문
- 하이퍼파라미터 값의 '좋음'을 시험 데이터로 확인하게 되므로 하이퍼파라미터의 값이 시험 데이터에만 적합헤도록 조정되어 버림
- 그러면 다른 데이터에는 적용하지 못하기 때문에 범용 성능이 떨어지는 모델이 될지도 모름

- 하이퍼파리미터를 조정할 때는 하이퍼파라미터 전용 확인 데이터가 필요

- 검증 데이터(validation data) : 하이퍼파라미터 조정용 데이터
- 하이퍼파라미터의 적절성을 평가하는 데이터인 셈

- 훈련 데이터는 매개변수(가중치와 편향)의 학습에 이용

- 검증 데이터는 하이퍼파라미터의 성능을 평가하는 데 이용
- 시험 데이터는 범용 성능을 확인하기 위해서 마지막에 (이상적으로는 한 번만) 이용

- 훈련 데이터 : 매개변수 학습

- 검증 데이터 : 하이퍼파라미터 성능 평가
- 시험 데이터 : 신경망의 범용 성능 평가

In [4]:
from tensorflow.keras.datasets import mnist
import numpy as np

(x_train, t_train), (x_test, t_test) = mnist.load_data()

# 데이터 섞기
shuffle_index = np.random.permutation(x_train.shape[0])
x_train = x_train[shuffle_index]
t_train = t_train[shuffle_index]

# 검증 데이터 분리
validation_rate = 0.20
validation_num = int(x_train.shape[0] * validation_rate)

x_val = x_train[:validation_num]
t_val = t_train[:validation_num]
x_train = x_train[validation_num:]
t_train = t_train[validation_num:]

#### 하이퍼파라미터 최적화

- 하이퍼파라미터를 최적화할 때의 핵심은 하이퍼파라미터의 '최적 값'이 존재하는 범위를 조금씩 줄여가는 것

- 범위를 조금씩 줄이려면 우선 대략적인 범위를 설정하고 그 범위에서 무작위로 하이퍼파라미터 값을 골라낸(샘플링) 후, 그 값으로 정확도를 평가

- 정확도를 잘 살피면서 이 작업을 여러 번 반복하여 하이퍼파라미터의 '최적 값'의 범위를 좁혀가는 것

- 신경망의 하이퍼파라미터 최적화에서는 그리드 서치(grid search)같은 규칙적인 탐색보다는 무작위로 탐색하는 것이 좋은 결과를 낸다고 알려져 있음

- 이는 최종 정확도에 미치는 영향력이 하이퍼파라미터마다 다르기 때문

- 하이퍼파라미터의 범위는 '대략적으로' 지정하는 것이 효과적

- 실제로도 0.001에서 1,000 사이와 같이 '10의 거듭제곱' 단위로 범위를 지정함
- 이를 '로그 스케일(log scale)로 지정'한다고 함

- 하이퍼파라미터를 최적화할 때 딥러닝 학습에는 오랜 시간이 걸림

- 학습을 위한 에폭을 작게 하여, 1회 평가에 걸리는 시간을 단축하는 것이 효과적

1. 하이퍼파라미터 값의 범위를 설정

2. 설정된 범위에서 하이퍼파라미터의 값을 무작위로 추출
3. 2에서 샘플링한 하이퍼파라미터 값을 사용하여 학습하고, 검증 데이터로 정확도를 평가 (단, 에폭은 작게 설정)

4. 2와 3을 특정 횟수 반복하여, 그 정확도의 결과를 보고 하이퍼파라미터의 범위를 좁힘

- 이상을 반복하여 하이퍼파라미터의 범위를 좁혀가고, 어느 정도 좁아지면 그 압축한 범위에서 값을 하나 골라냄

#### 하이퍼파라미터 최적화 구현하기

In [5]:
weight_decay = 10 ** np.random.uniform(-8, -4)
lr = 10 ** np.random.uniform(-6, -2)

- 이렇게 무작위로 추출한 값을 사용하여 학습을 수행

- 그 후에는 여러 차례 다양한 하이퍼파라미터 값으로 학습을 반복하며 신경망에 좋을 것 같은 값이 어디에 존재하는지 관찰

- 잘될 것 같은 값의 범위를 관찰하고 범위를 좁혀감

- 그런 다음 그 축소된 범위로 똑같은 작업을 반복
- 이렇게 적절한 값이 위치한 범위를 좁혀가다가 특정 단계에서 최종 하이퍼파라미터 값을 하나 선택

### 정리

- 매개변수 갱신 방법에는 확률적 경사 하강법(SGD) 외에도 모멘텀, AdaGrad, Adam 등이 있음

- 가중치 초깃값을 정하는 방법은 올바른 학습을 하는 데 매우 중요함
- 가중치의 초깃값으로는 'Xavier 초깃값'과 'He 초깃값'이 효과적
- 배치 정규화를 이용하면 학습을 빠르게 진행할 수 있으며, 초깃값에 영향을 덜 받게 됨
- 오버피팅을 억제하는 정규화 기술로는 가중치 감소와 드롭아웃이 있다
- 하이퍼파라미터 값 탐색은 최적 값이 존재할 법한 범위를 점차 좁히면서 하는 것이 효과적